# Phase 4: Portfolio Construction & Risk Management  
## Notebook 04_13 — Stochastic Control Portfolio Problem

### Research question

How can portfolio allocation be treated as a sequential decision problem, where today’s position affects future wealth, future opportunities, turnover, and utility?

This notebook follows:

```text
04_04_mean_variance_optimization_ledoit_wolf
04_09_risk_parity_and_equal_risk_contribution
04_11_cvar_convex_optimization
04_12_black_litterman_allocation
```

The previous notebooks solved mostly **single-period** allocation problems. This notebook moves to **multi-period stochastic control**.

It covers:

1. the portfolio problem as stochastic control;
2. dynamic programming and Bellman equations;
3. HJB intuition;
4. the continuous-time Merton benchmark;
5. CRRA utility;
6. regime-switching return dynamics;
7. discrete wealth grid;
8. discrete action grid;
9. transaction-cost-aware Bellman recursion;
10. optimal policy extraction;
11. policy visualisation;
12. Monte Carlo policy validation;
13. comparison with fixed-weight, Merton-style, cash, and regime heuristic policies;
14. certainty equivalent;
15. drawdown and tail-risk diagnostics;
16. sensitivity to risk aversion;
17. sensitivity to transaction costs;
18. governance checks;
19. saved outputs and manifest.

The key idea:

> Portfolio construction is not only “what weights are optimal now?” It is “what action should I take now, given how it changes future wealth and future decisions?”

## 1. Portfolio choice as stochastic control

Let wealth be $W_t$, market state be $S_t$, and portfolio action be $a_t$.

The investor chooses a policy:

$$
a_t = \pi(t,W_t,S_t)
$$

to maximise expected utility:

$$
\max_{\pi} E[U(W_T)]
$$

subject to wealth dynamics:

$$
\begin{aligned}
W_{t+1} &= W_t \Big[ 1+r_f+a_t(R_{t+1}-r_f) \Big] \\
&\quad - \text{cost}(a_t,a_{t-1},W_t)
\end{aligned}
$$

The control $a_t$ is the risky-asset weight.

The state includes:

1. time $t$;
2. current wealth $W_t$;
3. market regime $S_t$;
4. previous action $a_{t-1}$, if transaction costs matter.

This is a dynamic optimisation problem.

## 2. Bellman equation

The value function is:

$$
V_t(W,S,a_{prev}) = \max_a E_t[ V_{t+1}(W',S',a) ]
$$

where:

$$
\begin{aligned}
W' &= W \\
&\quad - c|a-a_{prev}|W
\end{aligned}
$$

then invested:

$$
W' \leftarrow W' \Big[ 1+r_f+a(R_{risky}-r_f) \Big]
$$

Terminal condition:

$$
V_T(W,S,a_{prev})=U(W)
$$

This notebook solves a discretised version of this Bellman recursion.

## 3. HJB intuition

In continuous time, the dynamic programming limit becomes a Hamilton-Jacobi-Bellman equation.

For a simple one-risky-asset wealth process:

$$
\begin{aligned}
dW_t &= \Big[ rW_t+\pi_tW_t(\mu-r) \Big]dt \\
&\quad + \pi_tW_t\sigma dB_t
\end{aligned}
$$

the HJB is:

$$
\begin{aligned}
0 &= \max_{\pi} \Big\{ V_t \\
&\quad + (rW+\pi W(\mu-r))V_W \\
&\quad + \frac{1}{2}\pi^2W^2\sigma^2V_{WW} \Big\}
\end{aligned}
$$

For CRRA utility, this leads to the Merton fraction:

$$
\pi^* = \frac{\mu-r}{\gamma\sigma^2}
$$

where $\gamma$ is risk aversion.

This elegant closed form breaks once we add constraints, transaction costs, regimes, drawdown controls, or discrete trading.

## 4. CRRA utility

Constant Relative Risk Aversion utility:

$$
U(W)= \frac{W^{1-\gamma}}{1-\gamma}
$$

for:

$$
\gamma \ne 1
$$

and log utility for:

$$
\gamma = 1
$$

Higher $\gamma$ means more risk aversion.

Certainty equivalent wealth $CE$ is the guaranteed wealth giving the same expected utility:

$$
U(CE)=E[U(W_T)]
$$

For CRRA:

$$
CE= \Big[ (1-\gamma)E[U(W_T)] \Big]^{\frac{1}{1-\gamma}}
$$

## 5. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class StochasticControlConfig:
    horizon_steps: int = 30
    steps_per_year: int = 12
    gamma: float = 4.0
    risk_free_rate_ann: float = 0.025
    risky_mean_calm_ann: float = 0.085
    risky_vol_calm_ann: float = 0.145
    risky_mean_stress_ann: float = -0.080
    risky_vol_stress_ann: float = 0.320
    transition_calm_to_stress: float = 0.055
    transition_stress_to_calm: float = 0.220
    min_wealth: float = 0.20
    max_wealth: float = 3.50
    n_wealth_grid: int = 140
    min_risky_weight: float = -0.25
    max_risky_weight: float = 1.50
    n_action_grid: int = 31
    transaction_cost_rate: float = 0.0015
    n_return_quadrature: int = 7
    n_sim_paths: int = 8_000
    initial_wealth: float = 1.0
    initial_regime: int = 0
    initial_action: float = 0.0
    seed: int = 42


config = StochasticControlConfig()
config

## 6. Regime-switching return model

We use two regimes:

| Regime | Meaning |
|---|---|
| 0 | calm |
| 1 | stress |

Transition matrix:

$$
P=
\begin{bmatrix}
P(0\to0) & P(0\to1) \\
P(1\to0) & P(1\to1)
\end{bmatrix}
$$

The risky asset has different mean and volatility in each regime.

This makes the optimal policy state-dependent:

- more risky allocation in calm;
- lower or even short risky allocation in stress;
- caution when transaction costs make switching expensive.

In [ ]:
def transition_matrix(config):
    p01 = config.transition_calm_to_stress
    p10 = config.transition_stress_to_calm
    return np.array([
        [1.0 - p01, p01],
        [p10, 1.0 - p10],
    ])


P_regime = transition_matrix(config)

pd.DataFrame(P_regime, index=["calm", "stress"], columns=["calm_next", "stress_next"])

In [ ]:
def regime_return_parameters(config):
    dt = 1.0 / config.steps_per_year

    rf_step = (1.0 + config.risk_free_rate_ann) ** dt - 1.0

    params = {
        0: {
            "name": "calm",
            "mean_ann": config.risky_mean_calm_ann,
            "vol_ann": config.risky_vol_calm_ann,
        },
        1: {
            "name": "stress",
            "mean_ann": config.risky_mean_stress_ann,
            "vol_ann": config.risky_vol_stress_ann,
        },
    }

    for regime in params:
        mu_ann = params[regime]["mean_ann"]
        vol_ann = params[regime]["vol_ann"]
        params[regime]["mu_step_log"] = (mu_ann - 0.5 * vol_ann**2) * dt
        params[regime]["vol_step_log"] = vol_ann * np.sqrt(dt)
        params[regime]["rf_step"] = rf_step

    return params


regime_params = regime_return_parameters(config)
regime_params

## 7. Quadrature approximation for returns

To compute expectations in the Bellman equation, we approximate normal shocks using Gauss-Hermite quadrature.

For:

$$
Z \sim N(0,1)
$$

we generate nodes $z_j$ and probabilities $p_j$.

Then risky simple return in regime $s$:

$$
R_s = \exp(\mu_s+\sigma_s z_j)-1
$$

In [ ]:
def normal_quadrature(n):
    nodes, weights = np.polynomial.hermite.hermgauss(n)

    # Convert Hermite nodes for exp(-x^2) to standard normal nodes.
    z = np.sqrt(2.0) * nodes
    p = weights / np.sqrt(np.pi)

    return z, p


quad_z, quad_p = normal_quadrature(config.n_return_quadrature)

pd.DataFrame({
    "z": quad_z,
    "probability": quad_p
})

In [ ]:
def quadrature_risky_returns(config, regime_params):
    rows = []

    for regime, params in regime_params.items():
        risky_returns = np.exp(params["mu_step_log"] + params["vol_step_log"] * quad_z) - 1.0

        for z, prob, ret in zip(quad_z, quad_p, risky_returns):
            rows.append({
                "regime": regime,
                "regime_name": params["name"],
                "z": z,
                "probability": prob,
                "risky_return": ret,
                "risk_free_return": params["rf_step"]
            })

    return pd.DataFrame(rows)


return_scenarios = quadrature_risky_returns(config, regime_params)
return_scenarios

## 8. Wealth and action grids

We discretise:

$$
W \in [W_{min},W_{max}]
$$

and risky weight:

$$
a \in [a_{min},a_{max}]
$$

The previous action is also discretised so transaction costs can be included in the state.

In [ ]:
wealth_grid = np.linspace(config.min_wealth, config.max_wealth, config.n_wealth_grid)
action_grid = np.linspace(config.min_risky_weight, config.max_risky_weight, config.n_action_grid)

grid_report = pd.Series({
    "wealth_min": wealth_grid.min(),
    "wealth_max": wealth_grid.max(),
    "n_wealth_grid": len(wealth_grid),
    "action_min": action_grid.min(),
    "action_max": action_grid.max(),
    "n_action_grid": len(action_grid),
})

grid_report

## 9. Utility and interpolation helpers

In [ ]:
def crra_utility(wealth, gamma):
    wealth = np.asarray(wealth, dtype=float)

    very_bad = -1e12

    out = np.empty_like(wealth, dtype=float)
    bad = wealth <= 0

    if abs(gamma - 1.0) < 1e-12:
        out[~bad] = np.log(wealth[~bad])
    else:
        out[~bad] = wealth[~bad] ** (1.0 - gamma) / (1.0 - gamma)

    out[bad] = very_bad
    return out


def certainty_equivalent(expected_utility, gamma):
    if abs(gamma - 1.0) < 1e-12:
        return float(np.exp(expected_utility))

    base = (1.0 - gamma) * expected_utility
    if base <= 0:
        return np.nan

    return float(base ** (1.0 / (1.0 - gamma)))


def interp_value(wealth_next, wealth_grid, value_grid):
    wealth_next = np.asarray(wealth_next, dtype=float)

    # If wealth goes outside the grid, clamp to edge values.
    # Negative/zero wealth is punished separately before interpolation.
    clipped = np.clip(wealth_next, wealth_grid[0], wealth_grid[-1])
    return np.interp(clipped, wealth_grid, value_grid)


terminal_utility = crra_utility(wealth_grid, config.gamma)

pd.Series({
    "terminal_utility_min": terminal_utility.min(),
    "terminal_utility_max": terminal_utility.max(),
})

## 10. Bellman solver

State:

$$
(t, regime, previous\ action, wealth)
$$

Control:

$$
a_t
$$

Transition:

1. pay transaction cost for changing from $a_{prev}$ to $a_t$;
2. realise risky return;
3. transition to next regime;
4. interpolate next value function.

The value recursion is:

$$
V_t = \max_a E[V_{t+1}]
$$

We store both value and optimal policy.

In [ ]:
def solve_discrete_stochastic_control(config):
    n_t = config.horizon_steps
    n_regimes = 2
    n_prev = len(action_grid)
    n_w = len(wealth_grid)
    n_a = len(action_grid)

    V = np.zeros((n_t + 1, n_regimes, n_prev, n_w), dtype=float)
    policy_idx = np.zeros((n_t, n_regimes, n_prev, n_w), dtype=int)

    # Terminal condition independent of regime and previous action.
    terminal = crra_utility(wealth_grid, config.gamma)
    for regime in range(n_regimes):
        for prev_idx in range(n_prev):
            V[n_t, regime, prev_idx, :] = terminal

    rf = regime_params[0]["rf_step"]

    scenario_returns_by_regime = {
        regime: return_scenarios[return_scenarios["regime"] == regime]["risky_return"].to_numpy()
        for regime in range(n_regimes)
    }

    scenario_probs = quad_p

    for t in range(n_t - 1, -1, -1):
        for regime in range(n_regimes):
            risky_returns = scenario_returns_by_regime[regime]

            for prev_idx, prev_action in enumerate(action_grid):
                best_values = np.full(n_w, -np.inf)
                best_actions = np.zeros(n_w, dtype=int)

                for action_idx, action in enumerate(action_grid):
                    transaction_cost = config.transaction_cost_rate * abs(action - prev_action) * wealth_grid
                    wealth_after_cost = wealth_grid - transaction_cost

                    expected_value = np.zeros(n_w)

                    for next_regime in range(n_regimes):
                        regime_prob = P_regime[regime, next_regime]
                        next_value_grid = V[t + 1, next_regime, action_idx, :]

                        scenario_value = np.zeros(n_w)

                        for ret, prob in zip(risky_returns, scenario_probs):
                            portfolio_return = rf + action * (ret - rf)
                            wealth_next = wealth_after_cost * (1.0 + portfolio_return)

                            values = interp_value(wealth_next, wealth_grid, next_value_grid)

                            # Bankruptcy penalty.
                            values = np.where(wealth_next <= 0, -1e12, values)
                            scenario_value += prob * values

                        expected_value += regime_prob * scenario_value

                    improve = expected_value > best_values
                    best_values[improve] = expected_value[improve]
                    best_actions[improve] = action_idx

                V[t, regime, prev_idx, :] = best_values
                policy_idx[t, regime, prev_idx, :] = best_actions

    policy_weights = action_grid[policy_idx]

    return V, policy_idx, policy_weights


V, policy_idx, policy_weights = solve_discrete_stochastic_control(config)

V.shape, policy_weights.shape

## 11. Initial value and certainty equivalent

We evaluate the value function at initial wealth, initial regime, and initial previous action.

In [ ]:
def nearest_index(array, value):
    array = np.asarray(array)
    return int(np.argmin(np.abs(array - value)))


initial_w_idx = nearest_index(wealth_grid, config.initial_wealth)
initial_prev_idx = nearest_index(action_grid, config.initial_action)
initial_regime = config.initial_regime

initial_expected_utility = V[0, initial_regime, initial_prev_idx, initial_w_idx]
initial_ce = certainty_equivalent(initial_expected_utility, config.gamma)
initial_policy_weight = policy_weights[0, initial_regime, initial_prev_idx, initial_w_idx]

pd.Series({
    "initial_expected_utility": initial_expected_utility,
    "initial_certainty_equivalent_wealth": initial_ce,
    "initial_optimal_risky_weight": initial_policy_weight,
    "initial_regime": initial_regime,
})

## 12. Policy visualisation

The optimal policy is state-dependent.

We plot:

$$
a^*(t,W,regime,a_{prev})
$$

for selected times and previous action values.

In [ ]:
def plot_policy_slice(t, regime, prev_action_value):
    prev_idx = nearest_index(action_grid, prev_action_value)
    weights = policy_weights[t, regime, prev_idx, :]

    plt.figure(figsize=(10, 6))
    plt.plot(wealth_grid, weights)
    plt.axhline(0, linestyle="--")
    plt.title(f"Optimal Risky Weight: t={t}, regime={regime}, previous action={action_grid[prev_idx]:.2f}")
    plt.xlabel("Wealth")
    plt.ylabel("Risky weight")
    plt.show()


for t in [0, config.horizon_steps // 2, config.horizon_steps - 1]:
    plot_policy_slice(t=t, regime=0, prev_action_value=0.0)
    plot_policy_slice(t=t, regime=1, prev_action_value=0.0)

## 13. Transaction-cost no-trade behaviour

With transaction costs, the previous action matters.

If current action is already close to optimal, the policy may avoid trading.

We show the policy at fixed wealth but different previous actions.

In [ ]:
wealth_value = 1.0
wealth_idx = nearest_index(wealth_grid, wealth_value)

policy_by_prev = []

for regime in [0, 1]:
    for t in [0, config.horizon_steps // 2]:
        for prev_idx, prev_action in enumerate(action_grid):
            policy_by_prev.append({
                "time": t,
                "regime": regime,
                "previous_action": prev_action,
                "optimal_action": policy_weights[t, regime, prev_idx, wealth_idx]
            })

policy_by_prev_df = pd.DataFrame(policy_by_prev)

policy_by_prev_df.head()

In [ ]:
for regime in [0, 1]:
    plt.figure(figsize=(10, 6))
    for t in [0, config.horizon_steps // 2]:
        sub = policy_by_prev_df[(policy_by_prev_df["regime"] == regime) & (policy_by_prev_df["time"] == t)]
        plt.plot(sub["previous_action"], sub["optimal_action"], marker="o", label=f"t={t}")
    plt.plot(action_grid, action_grid, linestyle="--", label="no trade line")
    plt.title(f"Policy as Function of Previous Action, Regime {regime}, Wealth={wealth_grid[wealth_idx]:.2f}")
    plt.xlabel("Previous action")
    plt.ylabel("Optimal new action")
    plt.legend()
    plt.show()

## 14. Continuous-time Merton benchmark

For a one-risky-asset Merton problem with CRRA utility:

$$
a^* = \frac{\mu-r}{\gamma\sigma^2}
$$

We compute calm, stress, and unconditional benchmarks.

The DP policy should be compared against this, not expected to equal it exactly because we have regimes, finite horizon, constraints, discrete actions, and transaction costs.

In [ ]:
def merton_fraction(mu_ann, rf_ann, vol_ann, gamma):
    return (mu_ann - rf_ann) / (gamma * vol_ann**2)


merton_calm = merton_fraction(
    config.risky_mean_calm_ann,
    config.risk_free_rate_ann,
    config.risky_vol_calm_ann,
    config.gamma
)

merton_stress = merton_fraction(
    config.risky_mean_stress_ann,
    config.risk_free_rate_ann,
    config.risky_vol_stress_ann,
    config.gamma
)

# Stationary regime probabilities.
p01 = config.transition_calm_to_stress
p10 = config.transition_stress_to_calm
pi_stress = p01 / (p01 + p10)
pi_calm = 1 - pi_stress

mu_uncond = pi_calm * config.risky_mean_calm_ann + pi_stress * config.risky_mean_stress_ann
second_moment_vol = np.sqrt(
    pi_calm * config.risky_vol_calm_ann**2
    + pi_stress * config.risky_vol_stress_ann**2
)
merton_uncond = merton_fraction(mu_uncond, config.risk_free_rate_ann, second_moment_vol, config.gamma)

merton_report = pd.Series({
    "stationary_calm_probability": pi_calm,
    "stationary_stress_probability": pi_stress,
    "merton_calm_fraction": merton_calm,
    "merton_stress_fraction": merton_stress,
    "merton_unconditional_fraction": merton_uncond,
    "clipped_merton_unconditional_fraction": np.clip(merton_uncond, config.min_risky_weight, config.max_risky_weight),
})

merton_report

## 15. Simulation engine for policies

We simulate wealth paths under several policies:

1. cash only;
2. fixed 60% risky;
3. unconditional Merton fraction;
4. regime heuristic;
5. stochastic-control policy.

Each simulation includes transaction costs.

In [ ]:
def simulate_regime_paths(config, n_paths, seed):
    rng = np.random.default_rng(seed)
    T = config.horizon_steps

    regimes = np.zeros((n_paths, T + 1), dtype=int)
    regimes[:, 0] = config.initial_regime

    risky_returns = np.zeros((n_paths, T))

    for t in range(T):
        current_regime = regimes[:, t]

        for regime in [0, 1]:
            idx = np.where(current_regime == regime)[0]
            if len(idx) == 0:
                continue

            params = regime_params[regime]
            z = rng.standard_normal(len(idx))
            risky_returns[idx, t] = np.exp(params["mu_step_log"] + params["vol_step_log"] * z) - 1.0

            probs = P_regime[regime]
            regimes[idx, t + 1] = rng.choice([0, 1], size=len(idx), p=probs)

    return regimes, risky_returns


def policy_cash(t, wealth, regime, prev_action):
    return 0.0


def policy_fixed_60(t, wealth, regime, prev_action):
    return 0.60


def policy_merton(t, wealth, regime, prev_action):
    return float(np.clip(merton_uncond, config.min_risky_weight, config.max_risky_weight))


def policy_regime_heuristic(t, wealth, regime, prev_action):
    if regime == 0:
        return 0.85
    return 0.10


def policy_dynamic_programming(t, wealth, regime, prev_action):
    w_idx = nearest_index(wealth_grid, wealth)
    prev_idx = nearest_index(action_grid, prev_action)
    return float(policy_weights[t, regime, prev_idx, w_idx])


def simulate_policy(config, policy_fn, seed):
    regimes, risky_returns = simulate_regime_paths(config, config.n_sim_paths, seed)

    T = config.horizon_steps
    wealth = np.zeros((config.n_sim_paths, T + 1))
    actions = np.zeros((config.n_sim_paths, T))
    costs = np.zeros((config.n_sim_paths, T))
    portfolio_returns = np.zeros((config.n_sim_paths, T))

    wealth[:, 0] = config.initial_wealth
    prev_action = np.full(config.n_sim_paths, config.initial_action)
    rf = regime_params[0]["rf_step"]

    for t in range(T):
        for i in range(config.n_sim_paths):
            action = policy_fn(t, wealth[i, t], regimes[i, t], prev_action[i])
            action = float(np.clip(action, config.min_risky_weight, config.max_risky_weight))

            cost = config.transaction_cost_rate * abs(action - prev_action[i]) * wealth[i, t]
            wealth_after_cost = max(wealth[i, t] - cost, 0.0)

            ret = rf + action * (risky_returns[i, t] - rf)
            wealth[i, t + 1] = wealth_after_cost * (1.0 + ret)

            actions[i, t] = action
            costs[i, t] = cost
            portfolio_returns[i, t] = ret

            prev_action[i] = action

    return {
        "wealth": wealth,
        "actions": actions,
        "costs": costs,
        "portfolio_returns": portfolio_returns,
        "regimes": regimes,
        "risky_returns": risky_returns,
    }


policy_functions = {
    "cash": policy_cash,
    "fixed_60": policy_fixed_60,
    "merton_unconditional": policy_merton,
    "regime_heuristic": policy_regime_heuristic,
    "dynamic_programming": policy_dynamic_programming,
}

simulation_results = {}

for name, fn in policy_functions.items():
    simulation_results[name] = simulate_policy(config, fn, seed=config.seed + 100 + len(simulation_results))

list(simulation_results.keys())

## 16. Simulation diagnostics

We evaluate terminal wealth, utility, certainty equivalent, turnover, cost, and tail risk.

In [ ]:
def path_drawdown(wealth_path):
    running_max = np.maximum.accumulate(wealth_path)
    dd = wealth_path / np.maximum(running_max, 1e-12) - 1.0
    return dd.min(axis=1)


def summarise_simulation_results(simulation_results, config):
    rows = []

    for name, result in simulation_results.items():
        wealth = result["wealth"]
        terminal = wealth[:, -1]
        util = crra_utility(terminal, config.gamma)
        expected_utility = float(np.mean(util))
        ce = certainty_equivalent(expected_utility, config.gamma)
        dd = path_drawdown(wealth)

        total_cost = result["costs"].sum(axis=1)
        action_turnover = np.abs(np.diff(np.c_[np.full(config.n_sim_paths, config.initial_action), result["actions"]], axis=1)).sum(axis=1)

        rows.append({
            "policy": name,
            "mean_terminal_wealth": float(np.mean(terminal)),
            "median_terminal_wealth": float(np.median(terminal)),
            "p05_terminal_wealth": float(np.quantile(terminal, 0.05)),
            "p01_terminal_wealth": float(np.quantile(terminal, 0.01)),
            "probability_terminal_loss": float(np.mean(terminal < config.initial_wealth)),
            "expected_utility": expected_utility,
            "certainty_equivalent": ce,
            "mean_max_drawdown": float(np.mean(dd)),
            "p05_max_drawdown": float(np.quantile(dd, 0.05)),
            "mean_total_cost": float(np.mean(total_cost)),
            "mean_action_turnover": float(np.mean(action_turnover)),
            "mean_average_risky_weight": float(np.mean(result["actions"])),
            "p95_abs_risky_weight": float(np.quantile(np.abs(result["actions"]), 0.95)),
        })

    return pd.DataFrame(rows).sort_values("certainty_equivalent", ascending=False)


simulation_summary = summarise_simulation_results(simulation_results, config)

simulation_summary

In [ ]:
plt.figure(figsize=(12, 6))
for name, result in simulation_results.items():
    avg_wealth = result["wealth"].mean(axis=0)
    plt.plot(np.arange(config.horizon_steps + 1), avg_wealth, label=name)
plt.title("Average Wealth Path by Policy")
plt.xlabel("Time step")
plt.ylabel("Average wealth")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
for name, result in simulation_results.items():
    plt.hist(result["wealth"][:, -1], bins=80, alpha=0.45, density=True, label=name)
plt.axvline(config.initial_wealth, linestyle="--")
plt.title("Terminal Wealth Distribution")
plt.xlabel("Terminal wealth")
plt.ylabel("Density")
plt.legend()
plt.show()

## 17. Policy action diagnostics

Compare average risky allocation through time.

In [ ]:
action_summary_frames = []

for name, result in simulation_results.items():
    avg_action = result["actions"].mean(axis=0)
    p10_action = np.quantile(result["actions"], 0.10, axis=0)
    p90_action = np.quantile(result["actions"], 0.90, axis=0)

    df = pd.DataFrame({
        "time": np.arange(config.horizon_steps),
        "policy": name,
        "avg_action": avg_action,
        "p10_action": p10_action,
        "p90_action": p90_action,
    })
    action_summary_frames.append(df)

action_summary = pd.concat(action_summary_frames, ignore_index=True)

action_summary.head()

In [ ]:
plt.figure(figsize=(12, 6))
for name in policy_functions.keys():
    sub = action_summary[action_summary["policy"] == name]
    plt.plot(sub["time"], sub["avg_action"], label=name)
plt.axhline(0, linestyle="--")
plt.title("Average Risky Weight Through Time")
plt.xlabel("Time step")
plt.ylabel("Average risky weight")
plt.legend()
plt.show()

## 18. Regime-conditioned actions

A successful state-dependent policy should behave differently in calm and stress regimes.

In [ ]:
regime_action_rows = []

for name, result in simulation_results.items():
    actions = result["actions"]
    regimes = result["regimes"][:, :-1]

    for regime in [0, 1]:
        mask = regimes == regime
        regime_action_rows.append({
            "policy": name,
            "regime": regime,
            "mean_action": float(actions[mask].mean()) if mask.sum() else np.nan,
            "median_action": float(np.median(actions[mask])) if mask.sum() else np.nan,
            "p10_action": float(np.quantile(actions[mask], 0.10)) if mask.sum() else np.nan,
            "p90_action": float(np.quantile(actions[mask], 0.90)) if mask.sum() else np.nan,
            "n_observations": int(mask.sum()),
        })

regime_action_report = pd.DataFrame(regime_action_rows)

regime_action_report

## 19. Tail-risk comparison

We compare policies using terminal wealth loss metrics.

Loss relative to initial wealth:

$$
L = 1-W_T
$$

Then compute VaR and CVaR.

In [ ]:
def historical_var_cvar(losses, alpha):
    losses = np.asarray(losses, dtype=float)
    var = np.quantile(losses, alpha)
    tail = losses[losses >= var]
    cvar = tail.mean() if len(tail) else np.nan
    return float(var), float(cvar)


tail_rows = []

for name, result in simulation_results.items():
    terminal = result["wealth"][:, -1]
    losses = config.initial_wealth - terminal

    for alpha in [0.95, 0.99]:
        var, cvar = historical_var_cvar(losses, alpha)
        tail_rows.append({
            "policy": name,
            "alpha": alpha,
            "terminal_loss_VaR": var,
            "terminal_loss_CVaR": cvar,
        })

tail_risk_report = pd.DataFrame(tail_rows)
tail_risk_report

In [ ]:
sub = tail_risk_report[tail_risk_report["alpha"] == 0.99].sort_values("terminal_loss_CVaR")

plt.figure(figsize=(10, 6))
plt.barh(sub["policy"], sub["terminal_loss_CVaR"])
plt.title("99% Terminal Loss CVaR")
plt.xlabel("CVaR of terminal loss")
plt.ylabel("Policy")
plt.show()

## 20. Sensitivity to risk aversion

Merton says:

$$
a^* \propto \frac{1}{\gamma}
$$

Higher risk aversion should reduce risky allocation.

Solving the full DP for many $\gamma$ values can be expensive, so this section compares Merton-style benchmarks and reports how the DP would be re-run.

In [ ]:
gamma_grid = [1.5, 2.0, 3.0, 4.0, 6.0, 8.0, 10.0]

gamma_sensitivity = []

for gamma in gamma_grid:
    calm = merton_fraction(config.risky_mean_calm_ann, config.risk_free_rate_ann, config.risky_vol_calm_ann, gamma)
    stress = merton_fraction(config.risky_mean_stress_ann, config.risk_free_rate_ann, config.risky_vol_stress_ann, gamma)
    uncond = merton_fraction(mu_uncond, config.risk_free_rate_ann, second_moment_vol, gamma)

    gamma_sensitivity.append({
        "gamma": gamma,
        "merton_calm": calm,
        "merton_stress": stress,
        "merton_unconditional": uncond,
        "clipped_unconditional": np.clip(uncond, config.min_risky_weight, config.max_risky_weight),
    })

gamma_sensitivity = pd.DataFrame(gamma_sensitivity)

gamma_sensitivity

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(gamma_sensitivity["gamma"], gamma_sensitivity["merton_calm"], marker="o", label="calm")
plt.plot(gamma_sensitivity["gamma"], gamma_sensitivity["merton_stress"], marker="x", label="stress")
plt.plot(gamma_sensitivity["gamma"], gamma_sensitivity["merton_unconditional"], marker="s", label="unconditional")
plt.axhline(0, linestyle="--")
plt.title("Merton Fraction Sensitivity to Risk Aversion")
plt.xlabel("Risk aversion gamma")
plt.ylabel("Risky weight")
plt.legend()
plt.show()

## 21. Sensitivity to transaction costs

Transaction costs create a no-trade region.

We do not re-run the full DP for every cost level here, but we simulate the fixed policies under alternative costs.

A full production study would re-solve the Bellman equation for each cost assumption.

In [ ]:
def simulate_policy_with_cost_override(config, policy_fn, transaction_cost_rate, seed):
    regimes, risky_returns = simulate_regime_paths(config, config.n_sim_paths, seed)

    T = config.horizon_steps
    wealth = np.zeros((config.n_sim_paths, T + 1))
    actions = np.zeros((config.n_sim_paths, T))
    costs = np.zeros((config.n_sim_paths, T))

    wealth[:, 0] = config.initial_wealth
    prev_action = np.full(config.n_sim_paths, config.initial_action)
    rf = regime_params[0]["rf_step"]

    for t in range(T):
        for i in range(config.n_sim_paths):
            action = policy_fn(t, wealth[i, t], regimes[i, t], prev_action[i])
            action = float(np.clip(action, config.min_risky_weight, config.max_risky_weight))

            cost = transaction_cost_rate * abs(action - prev_action[i]) * wealth[i, t]
            wealth_after_cost = max(wealth[i, t] - cost, 0.0)

            ret = rf + action * (risky_returns[i, t] - rf)
            wealth[i, t + 1] = wealth_after_cost * (1.0 + ret)

            actions[i, t] = action
            costs[i, t] = cost
            prev_action[i] = action

    return {"wealth": wealth, "actions": actions, "costs": costs, "regimes": regimes, "risky_returns": risky_returns}


cost_grid = [0.0, 0.0005, 0.0015, 0.0030, 0.0060]
cost_sensitivity_rows = []

for cost_rate in cost_grid:
    for name in ["fixed_60", "regime_heuristic", "dynamic_programming"]:
        fn = policy_functions[name]
        result = simulate_policy_with_cost_override(config, fn, cost_rate, seed=config.seed + int(cost_rate * 1e6) + len(name))
        terminal = result["wealth"][:, -1]
        util = crra_utility(terminal, config.gamma)
        ce = certainty_equivalent(float(util.mean()), config.gamma)

        cost_sensitivity_rows.append({
            "transaction_cost_rate": cost_rate,
            "policy": name,
            "mean_terminal_wealth": float(terminal.mean()),
            "certainty_equivalent": ce,
            "mean_total_cost": float(result["costs"].sum(axis=1).mean()),
            "mean_turnover": float(np.abs(np.diff(np.c_[np.full(config.n_sim_paths, config.initial_action), result["actions"]], axis=1)).sum(axis=1).mean()),
        })

cost_sensitivity = pd.DataFrame(cost_sensitivity_rows)

cost_sensitivity.head()

In [ ]:
plt.figure(figsize=(10, 6))
for policy in ["fixed_60", "regime_heuristic", "dynamic_programming"]:
    sub = cost_sensitivity[cost_sensitivity["policy"] == policy]
    plt.plot(sub["transaction_cost_rate"], sub["certainty_equivalent"], marker="o", label=policy)
plt.title("Certainty Equivalent Sensitivity to Transaction Costs")
plt.xlabel("Transaction cost rate")
plt.ylabel("Certainty equivalent wealth")
plt.legend()
plt.show()

## 22. Multi-asset Merton extension

In continuous time with multiple risky assets:

$$
w^* = \frac{1}{\gamma} \Sigma^{-1} (\mu-r\mathbf 1)
$$

This is useful as a local benchmark, but it is a single-period myopic rule under ideal assumptions.

We compute a small synthetic multi-asset Merton vector and show how easily it can become extreme.

In [ ]:
multi_assets = ["equity", "bond", "commodity", "trend"]
mu_ann = pd.Series({
    "equity": 0.075,
    "bond": 0.025,
    "commodity": 0.045,
    "trend": 0.050,
})
vol_ann = pd.Series({
    "equity": 0.16,
    "bond": 0.06,
    "commodity": 0.20,
    "trend": 0.10,
})
corr_multi = pd.DataFrame(
    [
        [1.00, -0.25, 0.25, -0.10],
        [-0.25, 1.00, 0.05, 0.15],
        [0.25, 0.05, 1.00, 0.05],
        [-0.10, 0.15, 0.05, 1.00],
    ],
    index=multi_assets,
    columns=multi_assets
)

cov_ann_multi = corr_multi.multiply(vol_ann, axis=0).multiply(vol_ann, axis=1)

excess_mu = mu_ann - config.risk_free_rate_ann
merton_multi = (1.0 / config.gamma) * np.linalg.pinv(cov_ann_multi.to_numpy()) @ excess_mu.to_numpy()

merton_multi_table = pd.DataFrame({
    "asset": multi_assets,
    "mu_ann": mu_ann.values,
    "vol_ann": vol_ann.values,
    "merton_weight_unconstrained": merton_multi,
})

merton_multi_table["merton_weight_clipped_long_only"] = np.clip(merton_multi_table["merton_weight_unconstrained"], 0, 1)
merton_multi_table["merton_weight_clipped_long_only"] /= merton_multi_table["merton_weight_clipped_long_only"].sum()

merton_multi_table

## 23. Governance flags

A stochastic-control allocation process should be reviewed if:

1. policy uses excessive leverage;
2. transaction costs dominate benefits;
3. certainty equivalent is worse than simple baselines;
4. tail CVaR is poor;
5. policy is too sensitive to regime assumptions;
6. DP grid boundaries bind too often;
7. solution is not interpretable.

In [ ]:
best_baseline_ce = simulation_summary[
    simulation_summary["policy"].isin(["cash", "fixed_60", "merton_unconditional", "regime_heuristic"])
]["certainty_equivalent"].max()

dp_row = simulation_summary[simulation_summary["policy"] == "dynamic_programming"].iloc[0]

governance_flags = pd.DataFrame([{
    "policy": "dynamic_programming",
    "certainty_equivalent": dp_row["certainty_equivalent"],
    "best_baseline_certainty_equivalent": best_baseline_ce,
    "p95_abs_risky_weight": dp_row["p95_abs_risky_weight"],
    "mean_total_cost": dp_row["mean_total_cost"],
    "terminal_loss_probability": dp_row["probability_terminal_loss"],
    "p01_terminal_wealth": dp_row["p01_terminal_wealth"],
    "flag_ce_below_best_baseline": bool(dp_row["certainty_equivalent"] < best_baseline_ce),
    "flag_p95_abs_weight_above_1_25": bool(dp_row["p95_abs_risky_weight"] > 1.25),
    "flag_mean_cost_above_2pct_wealth": bool(dp_row["mean_total_cost"] > 0.02),
    "flag_p01_terminal_wealth_below_0_7": bool(dp_row["p01_terminal_wealth"] < 0.70),
}])

governance_flags["review_required"] = governance_flags[
    [
        "flag_ce_below_best_baseline",
        "flag_p95_abs_weight_above_1_25",
        "flag_mean_cost_above_2pct_wealth",
        "flag_p01_terminal_wealth_below_0_7",
    ]
].any(axis=1)

governance_flags

## 24. Audit checks

Numerical checks:

1. transition probabilities sum to one;
2. quadrature probabilities sum to one;
3. value function is finite for valid wealth grid;
4. policy actions stay within bounds;
5. simulated wealth is finite and non-negative;
6. certainty equivalent is finite.

In [ ]:
audit_rows = []

audit_rows.append({
    "check": "transition_rows_sum_to_one",
    "value": float(np.max(np.abs(P_regime.sum(axis=1) - 1.0))),
    "passed": bool(np.max(np.abs(P_regime.sum(axis=1) - 1.0)) < 1e-12),
})

audit_rows.append({
    "check": "quadrature_probabilities_sum_to_one",
    "value": float(abs(quad_p.sum() - 1.0)),
    "passed": bool(abs(quad_p.sum() - 1.0) < 1e-12),
})

finite_value = bool(np.isfinite(V[V > -1e11]).all())
audit_rows.append({
    "check": "value_function_finite_on_nonbankrupt_states",
    "value": finite_value,
    "passed": finite_value,
})

policy_bounds_ok = bool((policy_weights >= config.min_risky_weight - 1e-12).all() and (policy_weights <= config.max_risky_weight + 1e-12).all())
audit_rows.append({
    "check": "policy_actions_within_bounds",
    "value": policy_bounds_ok,
    "passed": policy_bounds_ok,
})

all_sim_wealth_finite = all(np.isfinite(result["wealth"]).all() for result in simulation_results.values())
all_sim_wealth_nonnegative = all((result["wealth"] >= -1e-12).all() for result in simulation_results.values())

audit_rows.append({
    "check": "simulation_wealth_finite",
    "value": all_sim_wealth_finite,
    "passed": all_sim_wealth_finite,
})

audit_rows.append({
    "check": "simulation_wealth_nonnegative",
    "value": all_sim_wealth_nonnegative,
    "passed": all_sim_wealth_nonnegative,
})

ce_finite = bool(np.isfinite(simulation_summary["certainty_equivalent"].dropna()).all())
audit_rows.append({
    "check": "certainty_equivalents_finite",
    "value": ce_finite,
    "passed": ce_finite,
})

audit_report = pd.DataFrame(audit_rows)

audit_report

## 25. Practical checklist for stochastic control portfolios

Before trusting a stochastic-control policy:

1. **State definition**  
   Does the state include the variables that matter?

2. **Control bounds**  
   Are leverage and shorting constraints realistic?

3. **Transition model**  
   Are regime transitions plausible?

4. **Return model**  
   Are tails and stress regimes represented?

5. **Utility function**  
   Does CRRA utility match the investor objective?

6. **Transaction costs**  
   Are rebalancing costs included?

7. **Grid resolution**  
   Is the wealth/action grid fine enough?

8. **Boundary effects**  
   Does the policy hit grid limits?

9. **Policy interpretability**  
   Does the action make economic sense in each regime?

10. **Out-of-sample simulation**  
   Does it beat simple policies on certainty equivalent and tail risk?

11. **Sensitivity analysis**  
   Does the policy survive changes in $\gamma$, costs, and regimes?

12. **Governance**  
   Would a risk committee understand when it fails?

## 26. Saving outputs

The notebook saves:

1. configuration;
2. transition matrix;
3. return scenarios;
4. wealth grid;
5. action grid;
6. value-function slices;
7. policy slices;
8. policy-by-previous-action table;
9. Merton benchmark report;
10. simulation summary;
11. action summary;
12. regime action report;
13. terminal tail-risk report;
14. gamma sensitivity;
15. cost sensitivity;
16. multi-asset Merton table;
17. governance flags;
18. audit report;
19. manifest.

In [ ]:
output_dir = Path("data/processed/stochastic_control_portfolio_problem_v1")
output_dir.mkdir(parents=True, exist_ok=True)

config_path = output_dir / "config.json"
transition_path = output_dir / "transition_matrix.csv"
return_scenarios_path = output_dir / "quadrature_return_scenarios.csv"
wealth_grid_path = output_dir / "wealth_grid.csv"
action_grid_path = output_dir / "action_grid.csv"
value_slice_path = output_dir / "value_function_initial_slices.csv"
policy_slice_path = output_dir / "policy_initial_slices.csv"
policy_by_prev_path = output_dir / "policy_by_previous_action.csv"
merton_report_path = output_dir / "merton_report.csv"
simulation_summary_path = output_dir / "simulation_summary.csv"
action_summary_path = output_dir / "action_summary.csv"
regime_action_report_path = output_dir / "regime_action_report.csv"
tail_risk_report_path = output_dir / "terminal_tail_risk_report.csv"
gamma_sensitivity_path = output_dir / "gamma_sensitivity.csv"
cost_sensitivity_path = output_dir / "cost_sensitivity.csv"
merton_multi_path = output_dir / "multi_asset_merton_table.csv"
governance_flags_path = output_dir / "governance_flags.csv"
audit_report_path = output_dir / "audit_report.csv"
manifest_path = output_dir / "manifest.json"

Path(config_path).write_text(json.dumps(asdict(config), indent=2, default=str))
pd.DataFrame(P_regime, index=["calm", "stress"], columns=["calm_next", "stress_next"]).to_csv(transition_path)
return_scenarios.to_csv(return_scenarios_path, index=False)
pd.Series(wealth_grid, name="wealth").to_csv(wealth_grid_path, index=False)
pd.Series(action_grid, name="action").to_csv(action_grid_path, index=False)

# Save compact value/policy slices rather than the full 4D arrays.
slice_rows = []
policy_rows = []

for t in [0, config.horizon_steps // 2, config.horizon_steps - 1]:
    for regime in [0, 1]:
        for prev_action in [config.min_risky_weight, 0.0, 0.5, 1.0]:
            prev_idx = nearest_index(action_grid, prev_action)

            for w, val, pol in zip(
                wealth_grid,
                V[t, regime, prev_idx, :],
                policy_weights[t, regime, prev_idx, :]
            ):
                slice_rows.append({
                    "time": t,
                    "regime": regime,
                    "previous_action": action_grid[prev_idx],
                    "wealth": w,
                    "value": val,
                })
                policy_rows.append({
                    "time": t,
                    "regime": regime,
                    "previous_action": action_grid[prev_idx],
                    "wealth": w,
                    "optimal_action": pol,
                })

pd.DataFrame(slice_rows).to_csv(value_slice_path, index=False)
pd.DataFrame(policy_rows).to_csv(policy_slice_path, index=False)
policy_by_prev_df.to_csv(policy_by_prev_path, index=False)
merton_report.to_frame("value").to_csv(merton_report_path)
simulation_summary.to_csv(simulation_summary_path, index=False)
action_summary.to_csv(action_summary_path, index=False)
regime_action_report.to_csv(regime_action_report_path, index=False)
tail_risk_report.to_csv(tail_risk_report_path, index=False)
gamma_sensitivity.to_csv(gamma_sensitivity_path, index=False)
cost_sensitivity.to_csv(cost_sensitivity_path, index=False)
merton_multi_table.to_csv(merton_multi_path, index=False)
governance_flags.to_csv(governance_flags_path, index=False)
audit_report.to_csv(audit_report_path, index=False)

manifest = {
    "dataset_name": "stochastic_control_portfolio_problem_outputs",
    "schema_version": "stochastic_control_portfolio_problem_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "transition_matrix": pd.DataFrame(P_regime, index=["calm", "stress"], columns=["calm_next", "stress_next"]).to_dict(),
    "initial_expected_utility": initial_expected_utility,
    "initial_certainty_equivalent_wealth": initial_ce,
    "initial_optimal_risky_weight": initial_policy_weight,
    "merton_report": merton_report.to_dict(),
    "simulation_summary": simulation_summary.to_dict(orient="records"),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "notes": [
        "The dynamic programme uses state variables time, wealth, regime, and previous action.",
        "Transaction costs make the previous action part of the state.",
        "The risky return distribution is approximated by Gauss-Hermite quadrature.",
        "The policy is validated by Monte Carlo simulation against cash, fixed weight, Merton-style, and regime heuristic policies.",
        "The continuous-time Merton solution is included as a benchmark, not as a replacement for constrained dynamic control.",
        "This is an educational discrete stochastic-control implementation; production use requires richer state variables, calibration, and numerical validation."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

config_path, simulation_summary_path, policy_slice_path, governance_flags_path, manifest_path

## 27. Limitations

### 27.1 Low-dimensional state

The dynamic programme uses one risky asset and one regime state.

Real portfolios are multi-asset and high-dimensional.

### 27.2 Discretisation error

Wealth and action grids introduce approximation error.

A finer grid improves accuracy but increases runtime.

### 27.3 Return model risk

The regime-switching return model is synthetic.

Real regime transitions and return distributions are hard to estimate.

### 27.4 Utility model

CRRA utility is mathematically convenient, but real investors may have drawdown, liability, or benchmark-relative objectives.

### 27.5 Transaction cost simplification

Costs are proportional to turnover.

Real costs include spread, market impact, liquidity, financing, and taxes.

### 27.6 No learning

The policy knows the model.

A real investor learns parameters over time.

### 27.7 No constraints beyond weight bounds

Real portfolios require leverage, margin, asset-class, regulatory, and liquidity constraints.

### 27.8 No continuous-time numerical PDE

The notebook explains HJB intuition but solves a discrete Bellman problem.

## 28. Research frontier and extensions

Important extensions include:

1. **Multi-asset stochastic control**  
   Approximate dynamic programming or reinforcement learning.

2. **Transaction-cost no-trade regions**  
   Continuous-time impulse control.

3. **Drawdown-constrained control**  
   Include running maximum wealth as a state variable.

4. **Liability-driven investing**  
   Optimise surplus relative to liabilities.

5. **Partial observation**  
   Hidden Markov regimes with filtering.

6. **Robust control**  
   Optimise under model ambiguity.

7. **Reinforcement learning**  
   Learn policies directly from simulated or historical environments.

8. **Policy gradient methods**  
   Continuous action spaces with neural policies.

9. **Scenario-tree stochastic programming**  
   Multi-period optimisation with branching scenarios.

10. **Chinese futures stochastic control**  
   Include margin, price limits, night sessions, contract rolls, and regime-switching volatility.

## 29. Suggested follow-up notebooks

This notebook naturally leads to:

1. **04_14_portfolio_constraints_margin_and_leverage**  
   Add real implementation constraints.

2. **05_01_vectorized_backtest_engine**  
   Turn policies into reusable backtests.

3. **05_06_walk_forward_model_validation**  
   Validate dynamic policies out of sample.

4. **06_09_reinforcement_learning_trading_environment**  
   Move from dynamic programming to RL.

5. **07_01_multi_asset_cta_research_pipeline**  
   Add dynamic sizing and regime-aware control.

6. **07_04_stochastic_control_capstone**  
   Multi-asset approximate dynamic programming for futures.

## 30. Summary

This notebook implemented a stochastic-control portfolio problem.

It showed how to:

1. formulate portfolio choice as a sequential control problem;
2. write the Bellman equation;
3. connect dynamic programming to the HJB equation;
4. compute the Merton benchmark;
5. define CRRA utility and certainty equivalent;
6. simulate regime-switching returns;
7. build wealth and action grids;
8. solve a transaction-cost-aware Bellman recursion;
9. extract state-dependent policies;
10. visualise no-trade behaviour;
11. simulate policies by Monte Carlo;
12. compare cash, fixed weight, Merton, regime heuristic, and dynamic-programming policies;
13. evaluate terminal wealth, utility, drawdown, and tail risk;
14. run sensitivity analysis for risk aversion and transaction costs;
15. create governance flags and audit checks;
16. save outputs and metadata.

The key computational takeaway:

> Dynamic programming turns portfolio allocation into a recursive optimisation over state variables and future value.

The key financial takeaway:

> Optimal allocation depends not only on expected return and volatility, but also on wealth, regime, transaction costs, constraints, and future opportunity.

## 31. Further reading

- Merton, "Lifetime Portfolio Selection under Uncertainty."
- Fleming and Soner, *Controlled Markov Processes and Viscosity Solutions.*
- Pham, *Continuous-time Stochastic Control and Optimization with Financial Applications.*
- Bertsekas, *Dynamic Programming and Optimal Control.*
- Campbell and Viceira, *Strategic Asset Allocation.*
- Literature on no-trade regions, transaction-cost control, robust control, and reinforcement learning for portfolio choice.